# ATML PA3 — Task 2

Enable a Kaggle GPU and Internet. Add Kaggle secrets `HF_TOKEN` and `PA3_HF_REPO` for a private Hugging Face dataset. This notebook embeds the Task 2 code and stores data and checkpoints under `results/task2/`; no GitHub push is required. Use one session at a time for the same repository.

Run the GPU smoke and data preflight before training. After a session timeout, start a new GPU session and rerun setup, smoke, data preparation, and the interrupted training or evaluation cell. The same code and package versions are required for recovery. Feature shards and 250-update checkpoints are saved remotely.


In [ ]:
import os, sys, subprocess, base64, io, zipfile
from pathlib import Path
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
project = Path('/kaggle/working/PA3_task2')
project.mkdir(parents=True, exist_ok=True)
payload = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(payload))) as archive:
    for member in archive.namelist():
        assert member.startswith('task2_vlm/') and '..' not in member
    archive.extractall(project)
os.chdir(project)
sys.path.insert(0, str(project))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.20.2', 'huggingface-hub==1.33.0', 'safetensors>=0.5.3', 'Pillow', 'pandas', 'matplotlib'], check=True)
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
os.environ['PA3_HF_REPO'] = secrets.get_secret('PA3_HF_REPO')
from task2_vlm.recovery import _client
_client()
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU'
print('GPU:', torch.cuda.get_device_name(0))
def run_stage(*args):
    subprocess.run([sys.executable, '-m', *map(str, args)], cwd=project, check=True)


## GPU smoke gate

Run the real Qwen connector and LoRA memory/gradient smoke before the full experiment.


In [ ]:
run_stage('task2_vlm.smoke')
smoke_passed = True


## Select data and cache visual features

Uses the official COCO and VQAv2 downloads. The data manifest and each feature shard are uploaded to private recovery storage. Interrupted shard generation can resume.


In [ ]:
assert globals().get('smoke_passed'), 'Run the GPU smoke first'
from task2_vlm.artifacts import restore, publish
from task2_vlm.recovery import _client
from task2_vlm.prepare import preparation_hash
import json
root = project / 'results/task2'
root.mkdir(parents=True, exist_ok=True)
manifest = root / 'manifest.json'
features = root / 'features'
if restore(project):
    print('Restored verified Task 2 data and feature cache')
else:
    from huggingface_hub import snapshot_download
    api, repo_id, token = _client()
    remote_files = set(api.list_repo_files(repo_id, repo_type='dataset'))
    if 'results/task2/manifest.json' in remote_files:
        snapshot_download(repo_id=repo_id, repo_type='dataset', token=token, local_dir=str(project), allow_patterns=['results/task2/manifest.json'])
        print('Restored selected-data manifest')
    else:
        raw = root / 'raw_annotations'
        run_stage('task2_vlm.download', 'annotations', '--directory', raw)
        run_stage('task2_vlm.prepare', '--annotations', raw, '--images', root / 'images', '--output', manifest)
        api.upload_file(repo_id=repo_id, repo_type='dataset', path_or_fileobj=str(manifest), path_in_repo='results/task2/manifest.json')
    assert json.loads(manifest.read_text())['preparation_source_sha256'] == preparation_hash(), 'Saved split was created with different preparation code'
    run_stage('task2_vlm.feature_cache', '--images', manifest, '--output', features)
    publish(project)
assert json.loads(manifest.read_text())['preparation_source_sha256'] == preparation_hash(), 'Saved split was created with different preparation code'
run_stage('task2_vlm.preflight', '--manifest', manifest, '--features', features)
data_ready = True


## Three training stages

Each stage restores the latest verified recovery checkpoint. Checkpoints are uploaded every 250 updates. Stage averages printed every 25 updates provide a hardware-specific time estimate; development evaluation and transfers add time.


In [ ]:
assert globals().get('smoke_passed') and globals().get('data_ready'), 'Run smoke and data preparation first'
run_stage('task2_vlm.train', '--manifest', manifest, '--features', features, '--until', '2250')


In [ ]:
assert globals().get('smoke_passed') and globals().get('data_ready'), 'Run smoke and data preparation first'
run_stage('task2_vlm.train', '--manifest', manifest, '--features', features, '--until', '3750')


In [ ]:
assert globals().get('smoke_passed') and globals().get('data_ready'), 'Run smoke and data preparation first'
run_stage('task2_vlm.train', '--manifest', manifest, '--features', features, '--until', '4500')


## Final evaluation

Each suite and image condition is saved independently.


In [ ]:
assert globals().get('data_ready'), 'Run the data preparation cell first'
run_stage('task2_vlm.final_eval', '--manifest', manifest, '--features', features)


## Tables, figures, and qualitative examples

Results are uploaded under `results/task2/`. The student writes the report and interpretation.


In [ ]:
assert globals().get('data_ready'), 'Run the data preparation cell first'
run_stage('task2_vlm.analysis', '--manifest', manifest, '--features', features)
